# 🎙 Anandi Park — Generate TTS Audio with IndicF5

This notebook generates high-quality Hindi and Marathi call audio using [IndicF5](https://github.com/AI4Bharat/IndicF5).

**Steps:**
1. Install IndicF5 (takes ~2 min)
2. Generate audio for your call scripts
3. Download the WAV files
4. Upload to VPS at `/opt/anandi-park/anandi/uploads/tts/`

**Runtime:** Make sure you select **GPU** runtime: Runtime → Change runtime type → T4 GPU

In [ ]:
# Step 1: Install IndicF5
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q soundfile numpy

In [ ]:
# Step 2: Load the model (downloads ~3GB on first run)
from transformers import AutoModel
import numpy as np
import soundfile as sf
import os

print("Loading IndicF5 model...")
model = AutoModel.from_pretrained("ai4bharat/IndicF5", trust_remote_code=True)
print("✅ Model loaded!")

In [ ]:
# Step 3: Define your call scripts
# Edit these to match your pitch!

SCRIPTS = {
    "pitch-hindi": {
        "text": (
            "नमस्ते! मैं Anandi Park से बोल रहा हूँ। "
            "वाघोली-बकोरी रोड, पुणे पर प्रीमियम NA प्लॉट्स उपलब्ध हैं। "
            "कीमत सिर्फ पंद्रह लाख से शुरू। RERA रजिस्टर्ड, क्लीयर टाइटल। "
            "चौरासी प्लॉट्स में से कुछ ही बचे हैं। "
            "अगर आपको साइट विज़िट करना है तो कृपया एक दबाइए। धन्यवाद!"
        ),
        "lang": "hi",
    },
    "pitch-marathi": {
        "text": (
            "नमस्कार! मी Anandi Park कडून बोलतोय। "
            "वाघोळी-बकोरी रोड, पुणे येथे प्रीमियम NA प्लॉट्स उपलब्ध आहेत। "
            "किंमत फक्त पंधरा लाखापासून सुरू. RERA नोंदणीकृत, स्पष्ट मालकी हक्क। "
            "चौऱ्यांशी प्लॉट्सपैकी काही शिल्लक आहेत। "
            "साइट भेट करायची असल्यास कृपया एक दाबा. धन्यवाद!"
        ),
        "lang": "mr",
    },
    "pitch-english": {
        "text": (
            "Hello! I am calling from Anandi Park. "
            "Premium NA plots are available on Wagholi Bakori Road, Pune. "
            "Prices start from just fifteen lakhs. RERA registered, clear titles. "
            "Out of eighty four plots, only a few remain. "
            "If you would like to schedule a site visit, please press one. Thank you!"
        ),
        "lang": "en",
    },
    "followup-hindi": {
        "text": (
            "नमस्ते! मैं Anandi Park की टीम से बोल रहा हूँ। "
            "कुछ दिन पहले आपने हमारे प्लॉट्स के बारे में जानकारी ली थी। "
            "इस हफ्ते फ्री साइट विज़िट का मौका है। "
            "अगर आप आना चाहते हैं तो एक दबाइए। "
            "हमारी टीम आपसे संपर्क करेगी। धन्यवाद!"
        ),
        "lang": "hi",
    },
}

print(f"Defined {len(SCRIPTS)} scripts:")
for name, s in SCRIPTS.items():
    print(f"  {name} ({s['lang']}) — {len(s['text'])} chars")

In [ ]:
# Step 4: Generate audio for each script
# Uses a default reference prompt from the IndicF5 repo.

os.makedirs("output", exist_ok=True)

# Download a reference audio if not already present.
REF_AUDIO = "ref_prompt.wav"
REF_TEXT = "नमस्ते मेरा नाम गीता है क्या यह आपसे बात करने का सही समय है"

if not os.path.exists(REF_AUDIO):
    # Use a Hindi prompt from the IndicF5 repo.
    import urllib.request
    url = "https://huggingface.co/ai4bharat/IndicF5/resolve/main/prompts/HIN_F_HAPPY_00001.wav"
    urllib.request.urlretrieve(url, REF_AUDIO)
    print(f"Downloaded reference audio: {REF_AUDIO}")

results = {}

for name, script in SCRIPTS.items():
    print(f"\nGenerating: {name}...")
    try:
        audio = model(
            script["text"],
            ref_audio_path=REF_AUDIO,
            ref_text=REF_TEXT,
        )

        # Normalise to float32.
        if isinstance(audio, np.ndarray):
            if audio.dtype == np.int16:
                audio = audio.astype(np.float32) / 32768.0
        else:
            audio = np.array(audio, dtype=np.float32)

        out_path = f"output/{name}.wav"
        sf.write(out_path, audio, samplerate=24000)
        duration = len(audio) / 24000
        results[name] = out_path
        print(f"  ✅ Saved: {out_path} ({duration:.1f}s, {os.path.getsize(out_path) / 1024:.0f} KB)")
    except Exception as e:
        print(f"  ❌ Failed: {e}")

print(f"\n{'='*50}")
print(f"Generated {len(results)} audio files in output/")

In [ ]:
# Step 5: Play a preview (click the play button)
from IPython.display import Audio, display

for name, path in results.items():
    print(f"\n🔊 {name}:")
    display(Audio(path))

In [ ]:
# Step 6: Download all files
# In Colab, this triggers a browser download dialog.
from google.colab import files

for name, path in results.items():
    files.download(path)
    print(f"Downloaded: {path}")

## 📤 Upload to VPS

After downloading the WAV files, upload them to your VPS:

```bash
# From your personal machine (not the corporate laptop!):
scp output/*.wav root@147.93.169.183:/opt/anandi-park/anandi/uploads/tts/
```

Or upload via GitHub (add to `uploads/tts/` in the repo).

Then trigger calls with:
```json
POST /api/v1/ai-calling/blast
{
  "script": "http://147.93.169.183:4000/uploads/tts/pitch-hindi.wav",
  "tag": "scraped",
  "limit": 10
}
```

The API detects it's a URL and uses `<Play>` instead of `<Speak>`.

## 🎭 Voice Cloning (Optional)

Want to use Yuvraj's or Rajan's actual voice? Record them saying one sentence (5-10 seconds),
save as `developer_voice.wav`, upload to Colab, then change `REF_AUDIO` and `REF_TEXT`:

```python
REF_AUDIO = "developer_voice.wav"
REF_TEXT = "<exact words spoken in the recording>"
```

Re-run Step 4 and the generated audio will sound like that person.